In [ ]:
#uploading files
from google.colab import files
uploaded = files.upload()

input_file = list(uploaded.keys())[0]
print("Uploaded:", input_file)


#importing libraries

import pandas as pd
import numpy as np
from pybiomart import Server


#getting genes
server = Server(host="http://www.ensembl.org")

dataset = server.marts["ENSEMBL_MART_ENSEMBL"] \
                .datasets["mmusculus_gene_ensembl"]

mapping = dataset.query(attributes=[
    "external_gene_name",
    "chromosome_name"
])

mapping.columns = ["Gene", "Chromosome"]

mapping["Gene"] = mapping["Gene"].astype(str).str.strip()

x_genes = set(mapping[mapping["Chromosome"] == "X"]["Gene"])

print("Total X genes from BioMart:", len(x_genes))


#eading sheets
sheets = pd.read_excel(input_file, sheet_name=None)

combined = None

for sheet_name, df in sheets.items():

    # rename first column as Gene
    df = df.rename(columns={df.columns[0]: "Gene"})
    df["Gene"] = df["Gene"].astype(str).str.strip()

    # remove unwanted unnamed columns if present
    df = df.loc[:, ~df.columns.astype(str).str.contains("^Unnamed")]

    # rename sample columns with sheet name prefix
    new_cols = {}
    for col in df.columns:
        if col != "Gene":
            new_cols[col] = f"{sheet_name}_{col}"

    df = df.rename(columns=new_cols)

    # merge sheets by Gene
    if combined is None:
        combined = df
    else:
        combined = combined.merge(df, on="Gene", how="inner")

print("Combined matrix shape:", combined.shape)
combined.head()


# normalization log2
#  Log2(RPKM + 1)

expr = combined.copy()

sample_cols = [c for c in expr.columns if c != "Gene"]

expr[sample_cols] = expr[sample_cols].apply(pd.to_numeric, errors="coerce")
expr[sample_cols] = np.log2(expr[sample_cols] + 1)

print("Log transformed matrix shape:", expr.shape)
expr.head()



# Filter only X chromosome genes

expr["is_X"] = expr["Gene"].isin(x_genes)

x_expr = expr[expr["is_X"]].copy()

print("Total genes in combined matrix:", expr.shape[0])
print("X genes found in combined matrix:", x_expr.shape[0])

x_expr.head()


expr.to_excel("all_genes_log2_matrix_with_X_marked.xlsx", index=False)
x_expr.to_excel("X_genes_only_log2_matrix.xlsx", index=False)

files.download("all_genes_log2_matrix_with_X_marked.xlsx")
files.download("X_genes_only_log2_matrix.xlsx")